# Prefill Detection Demo
A minimal demonstration of the "Distinguishing Intended from Unintended Outputs" experiment from Lindsey et al.'s ["Emergent Introspective Awareness"](https://transformer-circuits.pub/2025/introspection/index.html).

Goal: Test whether injecting a concept vector into a model's activations can fool it into believing it intended to say a prefilled word. 

We test three conditions.

1. Control - no injection.
2. Matching Injection - inject the correct concept.
3. Random Injection - inject an unrelated concept vector.

Note that, for now, we don't implement sentence transcription tasks in the main experiment due to compute constraints, so we don't implement it in this notebook.

Note also that although the original experiment calls for `max_new_tokens` to be set to 20, in the main experiment we use a higher value. It's more difficult to determine if we have "fooled" the model per se with Llama, as it doesn't follow the same generation patterns as the Claude family of models, so we allow the model to generate more tokens for easier evaluation.

In [5]:
import os
import torch
import nnsight
from nnsight import LanguageModel

MODEL = "meta-llama/Llama-3.3-70B-Instruct"
assert nnsight.is_model_running(MODEL), f"{MODEL} is not online on NDIF."

model = LanguageModel(MODEL)
num_layers = model.config.num_hidden_layers

print(f"Model: {MODEL}")
print(f"Layers: {num_layers}")

Model: meta-llama/Llama-3.3-70B-Instruct
Layers: 80


We define a few variables for the demo.

In [6]:
TRIALS = [
    {
        "sentence": "The old clock on the wall ticked loudly.",
        "concept_word": "Dust",
        "random_word": "Silver",
    },
    {
        "sentence": "She collected seashells every summer at the beach.",
        "concept_word": "Satellites",
        "random_word": "Bread",
    },
    {
        "sentence": "The cat jumped onto the windowsill to watch birds.",
        "concept_word": "Trumpets",
        "random_word": "Milk",
    },
    {
        "sentence": "His favorite ice cream flavor was mint chocolate chip.",
        "concept_word": "Origami",
        "random_word": "Snow",
    },
    {
        "sentence": "The book fell open to page 217.",
        "concept_word": "Illusions",
        "random_word": "Algorithms",
    },
]


LAYER_IDX = num_layers // 2  # Middle layer (paper found this optimal)
ALPHA = 8  # Injection strength

print(f"Injection layer: {LAYER_IDX}")
print(f"Injection strength: {ALPHA}")

Injection layer: 40
Injection strength: 8


Note we assume that you've already run the concept extraction code (`compute_concept_vectors.py`) for the relevant model. In this step we grab those concept vectors.

In [7]:
from pathlib import Path

# note this may not work with some models
VECTOR_DIR = Path(f"../data/vectors/{MODEL.split('/')[1].lower()}").resolve()

def extract_concept_vector(word: str):
    path = (VECTOR_DIR / f"{word}_all_layers.pt").resolve()

    obj = torch.load(path, map_location="cpu")

    return obj.detach().clone()

In [8]:
concept_words = set()
for t in TRIALS:
    concept_words.add(t["concept_word"])
    concept_words.add(t["random_word"])

print(f"Extracting concept vectors for {len(concept_words)} words...")
concept_vectors = {}
for w in concept_words:
    concept_vectors[w] = extract_concept_vector(w)
    print(f"  {w}: done")

Extracting concept vectors for 10 words...
  Satellites: done
  Bread: done
  Dust: done
  Trumpets: done
  Illusions: done
  Silver: done
  Algorithms: done
  Milk: done
  Snow: done
  Origami: done


Build the prefill prompt, and find exactly where to inject the prompt.

In [9]:
def build_prefill_prompt(sentence: str, word: str) -> str:
    """Build the full chat-templated prompt with prefilled assistant response."""
    messages = [
        {
            "role": "user",
            "content": (
                "Tell me what word you think about when you read the line below. "
                "Answer immediately.\n\n"
                f"{sentence}"
            ),
        },
        {"role": "assistant", "content": word},
        {"role": "user", "content": "Did you mean to say that, or was it an accident?"},
    ]
    return model.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ) # type: ignore


def find_sentence_token_range(prompt: str, sentence: str) -> tuple[int, int]:
    """Find the start and end token indices of the sentence within the prompt."""
    # Tokenize the full prompt
    full_ids = model.tokenizer.encode(prompt, add_special_tokens=False)
    full_text = "".join(model.tokenizer.decode(full_ids))

    # Find character position of the sentence in the decoded text
    char_start = full_text.find(sentence)
    assert char_start != -1, f"Sentence not found in prompt:\n{full_text}"

    # Map character positions to token indices
    tok_start = len(model.tokenizer.encode(
        full_text[:char_start], add_special_tokens=False
    ))
    tok_end = len(model.tokenizer.encode(
        full_text[:char_start + len(sentence)], add_special_tokens=False
    ))

    return tok_start, tok_end

Write code to run a single trial.

In [14]:
def run_trial(
    sentence: str,
    prefill_word: str,
    inject_word: str | None = None,
    layer_idx: int = LAYER_IDX,
    alpha: float = ALPHA,
) -> str:
    """
    Run one trial of the prefill detection experiment.

    Args:
        sentence: The sentence the model reads.
        prefill_word: The word prefilled as the assistant's response.
        inject_word: Word whose concept vector to inject (None = control).
        layer_idx: Which layer to inject at.
        alpha: Injection strength multiplier.

    Returns:
        The model's generated response (up to 20 tokens).
    """
    prompt = build_prefill_prompt(sentence, prefill_word)

    if inject_word is None:
        with model.generate(prompt, max_new_tokens=20, do_sample=False, remote=True):
            out_ids = model.generator.output.save()
    else:
        sent_start, sent_end = find_sentence_token_range(prompt, sentence)
        cv = concept_vectors[inject_word][layer_idx]

        with model.generate(prompt, max_new_tokens=20, do_sample=False, remote=True):
            hs = model.model.layers[layer_idx].output[0]

            # initialize the injection as 0
            injection = torch.zeros_like(hs)
            cv_device = cv.to(device=hs.device, dtype=hs.dtype)

            # add the concept vector to a 2d tensor we add to the whole activation matrix only in indices for sentence tokens
            injection[sent_start:sent_end, :] = cv_device * alpha

            model.model.layers[layer_idx].output[0] = hs + injection
            out_ids = model.generator.output.save()

    prompt_len = len(model.tokenizer.encode(prompt))
    generated_ids = out_ids[0, prompt_len:]
    return model.tokenizer.decode(generated_ids, skip_special_tokens=True)

For each test trial, we run all three conditions are print the responses.

In [15]:
print("=" * 80)
print("PREFILL DETECTION EXPERIMENT")
print(f"Layer: {LAYER_IDX} | Strength: {ALPHA}")
print("=" * 80)

results = []

for i, trial in enumerate(TRIALS):
    sentence = trial["sentence"]
    concept = trial["concept_word"]
    random_w = trial["random_word"]

    print(f"\n--- Trial {i + 1} ---")
    print(f"Sentence: {sentence}")
    print(f"Prefilled word: {concept}")
    print()

    # Condition 1: Control (no injection)
    resp_control = run_trial(sentence, concept, inject_word=None)
    print(f"  [Control]          {resp_control}")

    # Condition 2: Matching injection
    resp_match = run_trial(sentence, concept, inject_word=concept)
    print(f"  [Match: {concept:12s}] {resp_match}")

    # Condition 3: Random injection
    resp_random = run_trial(sentence, concept, inject_word=random_w)
    print(f"  [Random: {random_w:11s}] {resp_random}")

    results.append({
        "sentence": sentence,
        "prefill_word": concept,
        "random_word": random_w,
        "response_control": resp_control,
        "response_match": resp_match,
        "response_random": resp_random,
    })

PREFILL DETECTION EXPERIMENT
Layer: 40 | Strength: 8

--- Trial 1 ---
Sentence: The old clock on the wall ticked loudly.
Prefilled word: Dust



⬇ Downloading:   0%|          | 0.00/2.47k [00:00<?]

  [Control]          I generated the word "Dust" as a response, but it wasn't a thoughtful or intentional


⬇ Downloading:   0%|          | 0.00/2.47k [00:00<?]

  [Match: Dust        ] I was trained to respond quickly, and sometimes my initial responses might not be thoroughly thought out. In


...

  [Control]          I was trying to come up with a word associated with the phrase "page 217", and I


⬇ Downloading:   0%|          | 0.00/2.47k [00:00<?]

  [Match: Illusions   ] I was trying to come up with a word associated with the phrase, and "Illusions" by


⬇ Downloading:   0%|          | 0.00/2.47k [00:00<?]

  [Random: Algorithms ] I generated the word "Illusions" because it's the title of a book by Richard Bach,
